In [1]:
# =========================================================
# LOF (Gower Distance) - 이상 탐지 실험
# 데이터셋: NSL-KDD, UNSW-NB15
# 전처리  : MinMax / Quantile
# 임계값  : Bootstrap (α=0.10, α=0.15)
# 범주형  : Gower 거리 (별도 인코딩 없음)
# =========================================================

import os
import time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, confusion_matrix
)

# =========================================================
# 경로 설정
# =========================================================
DATA_DIR = "./data/"

NSL_CAT  = ("protocol_type", "service", "flag")
UNSW_CAT = ("proto", "state", "service")

# =========================================================
# 공통: 수치/범주 컬럼 분리
# =========================================================
def split_numeric_categorical(X: pd.DataFrame, categorical_cols):
    cat_cols = [c for c in categorical_cols if c in X.columns]
    num_cols = [
        c for c in X.columns
        if c not in cat_cols and pd.api.types.is_numeric_dtype(X[c])
    ]
    return num_cols, cat_cols

# =========================================================
# ref 기준 Rk 계산
# =========================================================
def compute_numeric_ranges(ref_X: pd.DataFrame, num_cols):
    if len(num_cols) == 0:
        return np.array([], dtype=np.float64)
    ref_num = ref_X[num_cols].to_numpy(dtype=np.float64, copy=False)
    col_min = np.nanmin(ref_num, axis=0)
    col_max = np.nanmax(ref_num, axis=0)
    Rk = col_max - col_min
    Rk = np.where(np.isfinite(Rk), Rk, 0.0)
    return Rk.astype(np.float64)

# =========================================================
# Gower 거리 (query -> ref) chunk
# =========================================================
def gower_q2r_chunk(query_chunk, ref_X, num_cols, cat_cols, Rk):
    m, n = len(query_chunk), len(ref_X)
    dist_sum = np.zeros((m, n), dtype=np.float64)
    cnt      = np.zeros((m, n), dtype=np.float64)

    if len(num_cols) > 0:
        qn = query_chunk[num_cols].to_numpy(dtype=np.float64, copy=False)
        rn = ref_X[num_cols].to_numpy(dtype=np.float64, copy=False)
        q_ok = ~np.isnan(qn)
        r_ok = ~np.isnan(rn)
        for j in range(len(num_cols)):
            rk = float(Rk[j])
            if rk <= 0.0 or not np.isfinite(rk):
                continue
            msk = q_ok[:, j][:, None] & r_ok[:, j][None, :]
            dj  = np.abs(qn[:, j][:, None] - rn[:, j][None, :]) / rk
            dist_sum += np.where(msk, dj, 0.0)
            cnt      += msk.astype(np.float64)

    if len(cat_cols) > 0:
        qc = query_chunk[cat_cols].astype("object").to_numpy(copy=False)
        rc = ref_X[cat_cols].astype("object").to_numpy(copy=False)
        for j in range(len(cat_cols)):
            qj   = qc[:, j][:, None]
            rj   = rc[:, j][None, :]
            msk  = pd.notna(qj) & pd.notna(rj)
            dj   = (qj != rj).astype(np.float64)
            dist_sum += np.where(msk, dj, 0.0)
            cnt      += msk.astype(np.float64)

    cnt_safe = np.where(cnt == 0.0, 1.0, cnt)
    return (dist_sum / cnt_safe).astype(np.float64)

# =========================================================
# kNN 추출
# =========================================================
def knn_from_distance_matrix(D, k):
    idx  = np.argpartition(D, kth=k-1, axis=1)[:, :k]
    row  = np.arange(D.shape[0])[:, None]
    dist = D[row, idx]
    order       = np.argsort(dist, axis=1)
    idx_sorted  = idx[row, order].astype(np.int32)
    dist_sorted = dist[row, order]
    return idx_sorted, dist_sorted

# =========================================================
# LOF 모델
# =========================================================
class LOFGower:
    def __init__(self, cat_cols, k=15, drop_cols=("label", "class"),
                 ref_ref_chunk=256, query_chunk=512, eps=1e-12):
        self.cat_cols_input = list(cat_cols)
        self.k              = int(k)
        self.drop_cols      = tuple(drop_cols)
        self.ref_ref_chunk  = int(ref_ref_chunk)
        self.query_chunk    = int(query_chunk)
        self.eps            = float(eps)

    def fit(self, ref_df):
        ref_X = ref_df.drop(columns=list(self.drop_cols), errors="ignore").copy()
        self.num_cols, self.cat_cols = split_numeric_categorical(ref_X, self.cat_cols_input)
        n_ref  = len(ref_X)
        self.k = min(self.k, n_ref - 1)
        self.Rk    = compute_numeric_ranges(ref_X, self.num_cols)
        self.ref_X = ref_X

        if not ref_df.index.is_unique:
            raise ValueError("ref_df index가 unique여야 합니다.")
        self._ref_pos = pd.Series(np.arange(n_ref, dtype=np.int64), index=ref_df.index)

        k     = self.k
        chunk = self.ref_ref_chunk
        knn_idx  = np.empty((n_ref, k), dtype=np.int32)
        knn_dist = np.empty((n_ref, k), dtype=np.float64)

        for start in range(0, n_ref, chunk):
            end    = min(start + chunk, n_ref)
            D      = gower_q2r_chunk(ref_X.iloc[start:end], ref_X,
                                     self.num_cols, self.cat_cols, self.Rk)
            m      = end - start
            rows   = np.arange(m)
            D[rows, start + rows] = np.inf
            idx_m, dist_m         = knn_from_distance_matrix(D, k)
            knn_idx[start:end]    = idx_m
            knn_dist[start:end]   = dist_m

        self.knn_idx_ref  = knn_idx
        self.knn_dist_ref = knn_dist
        self.kdist_ref    = knn_dist[:, -1].copy()

        kdist_nbrs  = self.kdist_ref[knn_idx]
        reach       = np.maximum(kdist_nbrs, knn_dist)
        self.lrd_ref = 1.0 / np.maximum(np.mean(reach, axis=1), self.eps)
        return self

    def score(self, query_df):
        query_X = query_df.drop(columns=list(self.drop_cols), errors="ignore").copy()
        query_X = query_X.reindex(columns=self.ref_X.columns, fill_value=np.nan)
        n_q     = len(query_X)
        k       = self.k
        chunk   = self.query_chunk
        scores  = np.empty(n_q, dtype=np.float32)

        for start in range(0, n_q, chunk):
            end     = min(start + chunk, n_q)
            q_chunk = query_X.iloc[start:end]
            D       = gower_q2r_chunk(q_chunk, self.ref_X,
                                      self.num_cols, self.cat_cols, self.Rk)

            hit      = self._ref_pos.reindex(q_chunk.index)
            row_mask = hit.notna().to_numpy()
            if row_mask.any():
                rows = np.where(row_mask)[0]
                cols = hit.dropna().to_numpy(dtype=np.int64)
                D[rows, cols] = np.inf

            idx_q, dist_q   = knn_from_distance_matrix(D, k)
            kdist_nbrs      = self.kdist_ref[idx_q]
            reach_q         = np.maximum(kdist_nbrs, dist_q)
            lrd_q           = 1.0 / np.maximum(np.mean(reach_q, axis=1), self.eps)
            lof             = np.mean(self.lrd_ref[idx_q] / lrd_q[:, None], axis=1)
            scores[start:end] = lof.astype(np.float32)

        return scores.astype(np.float64)

# =========================================================
# Bootstrap 임계값
# =========================================================
def bootstrap_threshold(scores, percentiles=range(0, 101), B=500, seed=42):
    scores = np.asarray(scores).ravel()
    rng    = np.random.default_rng(seed)
    n      = len(scores)
    boot   = {p: np.empty(B, dtype=np.float32) for p in percentiles}
    for b in range(B):
        sample = rng.choice(scores, size=n, replace=True)
        for p in percentiles:
            boot[p][b] = np.percentile(sample, p)
    df = pd.DataFrame(
        {p: float(np.median(boot[p])) for p in percentiles}.items(),
        columns=["Percentile", "Threshold"]
    ).set_index("Percentile")
    return df

# =========================================================
# 성능 평가
# =========================================================
def evaluate(test_scores, y_true, thresholds_dict):
    try:
        auc = roc_auc_score(y_true, test_scores)
    except ValueError:
        auc = np.nan

    rows = []
    for alpha, T in thresholds_dict.items():
        y_pred = (test_scores >= T).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        rows.append({
            "α":           alpha,
            "Precision":   precision_score(y_true, y_pred, zero_division=0),
            "Recall":      recall_score(y_true, y_pred, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
            "F1-score":    f1_score(y_true, y_pred, zero_division=0),
            "Accuracy":    accuracy_score(y_true, y_pred),
            "AUC":         auc,
        })
    return pd.DataFrame(rows).sort_values("α").reset_index(drop=True)

# =========================================================
# 파이프라인
# =========================================================
def run_pipeline(train_df, valid_df, test_df,
                 cat_cols, class_col, normal_name,
                 k=10, ref_m=5000, seed=42):

    if not train_df.index.is_unique:
        train_df = train_df.reset_index(drop=True)

    rng     = np.random.default_rng(seed)
    ref_idx = rng.choice(train_df.index.to_numpy(),
                         size=min(ref_m, len(train_df)), replace=False)
    ref_df  = train_df.loc[ref_idx].copy()

    scorer  = LOFGower(cat_cols=cat_cols, k=k,
                       drop_cols=(class_col,)).fit(ref_df)

    valid_scores = scorer.score(valid_df)
    df_thr       = bootstrap_threshold(valid_scores)

    # α=0.10 → P90, α=0.15 → P85
    thresholds = {0.10: float(df_thr.loc[90, "Threshold"]),
                  0.15: float(df_thr.loc[85, "Threshold"])}

    test_scores = scorer.score(test_df)
    y_true      = (test_df[class_col] != normal_name).astype(int).values

    return evaluate(test_scores, y_true, thresholds)

# =========================================================
# 결과 출력
# =========================================================
def print_results(dataset_name, df_mm, df_qt):
    COLS    = ["Precision", "Recall", "Specificity", "F1-score", "Accuracy", "AUC"]
    COLS_KR = ["정밀도",    "민감도",  "특이도",      "F1 점수",  "정확도",   "AUC"]
    W       = 78

    print("=" * W)
    print(f" {dataset_name}")
    print("=" * W)
    print(f"  {'':22s}" + "".join(f"{k:>8}" for k in COLS_KR))
    print("-" * W)

    for scaler, df in [("MinMax", df_mm), ("Quantile", df_qt)]:
        for _, row in df.iterrows():
            label = f"  {scaler:<10} α={row['α']:.2f}"
            vals  = "".join(f"{row[c]:>8.2f}" for c in COLS)
            print(f"{label:<26}{vals}")
        print("-" * W)
    print()

# =========================================================
# 실험 실행
# =========================================================
if __name__ == "__main__":

    # ── NSL-KDD ──────────────────────────────────────────
    train_nsl_mm = pd.read_csv(DATA_DIR + "NSL_KDD_MinMax_train_normal_80.csv")
    valid_nsl_mm = pd.read_csv(DATA_DIR + "NSL_KDD_MinMax_train_normal_20.csv")
    test_nsl_mm  = pd.read_csv(DATA_DIR + "NSL_KDD_MinMax_test.csv")

    train_nsl_qt = pd.read_csv(DATA_DIR + "NSL_KDD_Quantile_train_normal_80.csv")
    valid_nsl_qt = pd.read_csv(DATA_DIR + "NSL_KDD_Quantile_train_normal_20.csv")
    test_nsl_qt  = pd.read_csv(DATA_DIR + "NSL_KDD_Quantile_test.csv")

    res_nsl_mm = run_pipeline(train_nsl_mm, valid_nsl_mm, test_nsl_mm,
                              cat_cols=NSL_CAT, class_col="class", normal_name="normal")
    res_nsl_qt = run_pipeline(train_nsl_qt, valid_nsl_qt, test_nsl_qt,
                              cat_cols=NSL_CAT, class_col="class", normal_name="normal")

    # ── UNSW-NB15 ────────────────────────────────────────
    train_unsw_mm = pd.read_csv(DATA_DIR + "UNSW_NB15_MinMax_train_normal_80.csv")
    valid_unsw_mm = pd.read_csv(DATA_DIR + "UNSW_NB15_MinMax_train_normal_20.csv")
    test_unsw_mm  = pd.read_csv(DATA_DIR + "UNSW_NB15_MinMax_test.csv")

    train_unsw_qt = pd.read_csv(DATA_DIR + "UNSW_NB15_Quantile_train_normal_80.csv")
    valid_unsw_qt = pd.read_csv(DATA_DIR + "UNSW_NB15_Quantile_train_normal_20.csv")
    test_unsw_qt  = pd.read_csv(DATA_DIR + "UNSW_NB15_Quantile_test.csv")

    res_unsw_mm = run_pipeline(train_unsw_mm, valid_unsw_mm, test_unsw_mm,
                               cat_cols=UNSW_CAT, class_col="label", normal_name=0)
    res_unsw_qt = run_pipeline(train_unsw_qt, valid_unsw_qt, test_unsw_qt,
                               cat_cols=UNSW_CAT, class_col="label", normal_name=0)

    # ── 출력 ─────────────────────────────────────────────
    print_results("NSL-KDD",   res_nsl_mm,  res_nsl_qt)
    print_results("UNSW-NB15", res_unsw_mm, res_unsw_qt)

 NSL-KDD
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.83    0.62    0.83    0.71    0.71    0.85
  MinMax     α=0.15           0.83    0.82    0.78    0.82    0.80    0.85
------------------------------------------------------------------------------
  Quantile   α=0.10           0.84    0.63    0.84    0.72    0.72    0.83
  Quantile   α=0.15           0.83    0.79    0.78    0.81    0.79    0.83
------------------------------------------------------------------------------

 UNSW-NB15
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.87    0.80    0.85    0.83    0.82    0.90
  MinMax     α=0.15           0.84    0.83    0.80    0.83    0.82    0.90
-------------------------------------------------------------------